In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
from shapely import force_2d  # harmless if not 3D, available in Shapely 2.x
import pyogrio  # faster I/O (already in your logs)
import pandas as pd
from pathlib import Path


In [ ]:
# --- INPUTS: edit these 2 paths (and layer names if needed) ---
LAND_PATH   = "/Users/robynhaggis/Documents/land_mask_poly.gpkg"
LAND_LAYER  = None   # e.g. "land_mask_poly" if your GPKG has multiple layers


In [ ]:
BASINS_PATH  = "/Users/robynhaggis/Documents/major_river_basins_vector.gpkg"
BASINS_LAYER = "major_river_basins_vector"  # change if different


In [ ]:
OUT_GPKG = "/Users/robynhaggis/Documents/Geospatial_analysis/land_minus_basins.gpkg"


In [ ]:
# --- helpers ---
def first_layer(path):
    """Pick the first layer name if none provided."""
    lst = pyogrio.list_layers(path)
    # pyogrio returns a pandas DataFrame in newer versions, handle both
    if isinstance(lst, pd.DataFrame):
        return lst.iloc[0]["name"]
    else:
        # older pyogrio: list of tuples (name, geom_type, feature_count, extent, srs_wkt)
        return lst[0][0]

def read_layer(path, layer=None):
    if layer is None:
        layer = first_layer(path)
    return gpd.read_file(path, layer=layer, engine="pyogrio")

def fix_polys(gdf):
    # Keep only polygonal geometries and make them valid; buffer(0) is robust across versions
    gdf = gdf[gdf.geometry.notnull()].copy()
    # Drop non-polygons if any
    gdf = gdf[gdf.geom_type.isin(["Polygon","MultiPolygon"])].copy()
    gdf["geometry"] = gdf.geometry.buffer(0)
    # If any Z, drop Z to be safe
    try:
        gdf["geometry"] = gdf.geometry.apply(force_2d)
    except Exception:
        pass
    return gdf


In [ ]:
# --- read data ---
land   = read_layer(LAND_PATH, LAND_LAYER)
basins = read_layer(BASINS_PATH, BASINS_LAYER)


In [ ]:
# Fix invalids
land_fix   = fix_polys(land)
basins_fix = fix_polys(basins)


In [ ]:
# Dissolve all basins into a single coverage (very robust via unary_union)
basins_union_geom = unary_union(basins_fix.geometry.values)
basins_union = gpd.GeoDataFrame({"id":[1]}, geometry=[basins_union_geom], crs=land_fix.crs)

# Compute land minus basins (the gap polygons)
gaps = gpd.overlay(land_fix, basins_union, how="difference")


In [ ]:
gaps.plot()

In [ ]:
Path(OUT_GPKG).unlink(missing_ok=True)
gaps.to_file(OUT_GPKG, layer="catchment_polygons_coastal", driver="GPKG")


In [ ]:


SRC_GPKG = "/Users/robynhaggis/Documents/land_minus_basins.gpkg"
SRC_LAYER = "gaps"            # change if your layer is named differently
OUT_GPKG = "/Users/robynhaggis/Documents/land_minus_basins_single.gpkg"

gdf = gpd.read_file(SRC_GPKG, layer=SRC_LAYER)
gdf = gdf[gdf.geometry.notnull() & gdf.geom_type.isin(["Polygon","MultiPolygon"])].copy()

# explode multipolygons into single polygons
try:
    single = gdf.explode(index_parts=True, ignore_index=True)   # GeoPandas ≥0.10
except TypeError:
    single = gdf.explode().reset_index(drop=True)               # older GeoPandas

# add area fields (EPSG:3448 is metric → m²)
single["area_m2"]  = single.geometry.area
single["area_km2"] = single["area_m2"] / 1e6

# optional: drop tiny slivers
MIN_KM2 = 0.001
single = single[single["area_km2"] >= MIN_KM2].copy()

# give a simple id
single["gap_id"] = range(1, len(single) + 1)

Path(OUT_GPKG).unlink(missing_ok=True)
single.to_file(OUT_GPKG, layer="gaps_single", driver="GPKG")
print(f"Wrote {len(single)} polygons to {OUT_GPKG}")